### ООП

In [1]:
# __name__ __iter__ __next__ __hash__

In [2]:
# len(x) -> x.__len__(); a + b -> a.__add__(b); for -> __iter__

In [4]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

v = Vector(1, 2)
print(v)
print([v, v])

[<__main__.Vector object at 0x1081fae40>, <__main__.Vector object at 0x1081fae40>]


In [6]:
# __repr__ __str__

In [10]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f'Vector({self.x}, {self.y})'

    def __str__(self):
        return f'Вектор ({self.x}, {self.y})'

v = Vector(1, 2)
# repr print([v])
print([v, v])

print(v)
print(f'{v}')

[Vector(1, 2), Vector(1, 2)]
Вектор (1, 2)
Вектор (1, 2)


In [2]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f'Vector({self.x}, {self.y})'

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, k):
        if not isinstance(k, (int, float)):
            return NotImplemented
        return Vector(self.x * k, self.y * k)

    def __rmul__(self, k):
        return self * k

    def __abs__(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

    def __bool__(self):
        return bool(self.x or self.y)


a, b = Vector(1, 2), Vector(3, 4)
print(a + b, a * 3, 3 * a, abs(b))
print(a == Vector(1, 2), a == (1, 2), bool(Vector(0, 0)))


try:
    sum([a, b])
except TypeError as e:
    print('TypeError:', e)
print(sum([a, b], Vector(0, 0)))

Vector(4, 6) Vector(3, 6) Vector(3, 6) 5.0
True False False
TypeError: unsupported operand type(s) for +: 'int' and 'Vector'
Vector(4, 6)


In [4]:
Vector('a', None)

Vector(a, None)

### @property

In [5]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

acc = Account('Anna', 100)
acc.balance = -1_000_000
print(acc.balance)

-1000000


In [6]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError('баланс должен быть числом')
        if value < 0:
            raise ValueError('баланс не может быть отрицательным')
        self._balance = value

In [11]:
acc = Account('Anna', balance=100)
acc.balance += 50
print(acc.balance, acc.__dict__)
try:
    acc.balance = -5
except ValueError as e:
    print(e)

try:
    acc.balance = 'много'
except TypeError as e:
    print(e)
print()
try:
    acc_bad = Account('Anna', '-5')
except TypeError as e:
    print(e)

150 {'owner': 'Anna', '_balance': 150}
баланс не может быть отрицательным
баланс должен быть числом

баланс должен быть числом


In [17]:
class Rectangle:
    def __init__(self, w, h):
        self.w, self.h = w, h

    @property
    def area(self):
        return self.w * self.h

r = Rectangle(2, 3)
print(r.area)
r.w = 10
print(r.area)

# r.area = 100

r.__dict__['area'] = 'подделка'
print(r.area, r.__dict__)

6
30
30 {'w': 10, 'h': 3, 'area': 'подделка'}


### dataclasses

In [18]:
from dataclasses import dataclass
import inspect

In [27]:
class PointManual:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f'PointManual(x={self.x!r}, y={self.y!r})'

    def __eq__(self, other):
        if not isinstance(other, PointManual):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)



a, b = PointManual(1, 2), PointManual(1, 2)
print(a == b, hash(a) == hash(b))
d = {a: 'точка'}
print(b in d)
s = {a, b}
print(s)


print()
print()

@dataclass
class Point:
    x: float
    y: float

p = Point(1, 2)
print(p, Point(1, 2) == Point(1, 2))
print(inspect.signature(Point.__init__))
print(Point.__hash__)

True False
False
{PointManual(x=1, y=2), PointManual(x=1, y=2)}


Point(x=1, y=2) True
(self, x: float, y: float) -> None
None


In [30]:
p = Point('один', 'два')
p

Point(x='один', y='два')

In [31]:
from dataclasses import dataclass, field

try:
    @dataclass
    class Student:
        name: str
        grades: list[int] = []

except ValueError as e:
    print(e)

mutable default <class 'list'> for field grades is not allowed: use default_factory


In [34]:
from dataclasses import dataclass, field


@dataclass
class Student:
    name: str
    group: str = 'РТФ-1'
    grades: list[int] = field(default_factory=list)

a = Student(name='Anna')
print(a)

Student(name='Anna', group='РТФ-1', grades=[])


In [42]:
from dataclasses import dataclass, field

@dataclass
class Order:
    item: str
    price: float
    qty: int = 1
    total: float = field(init=False)

    def __post_init__(self):
        if self.price < 0:
            raise ValueError('Цена не может быть отрицательной')
        if self.qty <= 0:
            raise ValueError('количество должно быть положительным')
        self.total = self.price * self.qty

print(Order('book', 500, 2))
a = Order('book', 500, 2)
a.price = 450
a.total


Order(item='book', price=500, qty=2, total=1000)


1000

In [46]:
@dataclass(frozen=True)  # координаты, деньги, даты, настройки etc
class Point:
    x: float
    y: float

a = Point(1, 2)
b = Point(1, 2)
print({a, b})

{Point(x=1, y=2)}


In [48]:
# @dataclass(order=True) - генерирует <, <=, >=, >
# @dataclass(slots - объект без __dict__, kw_only - разрешает передавать только именнованные аргументы, asdict - превращает объект в словарь - например, а дальше можно сериализовать в JSON) - 

In [51]:
def greet(name: str, times: int = 1) -> str:
    return ', '.join([f'Привет, {name}'] * times)

print(greet('Anna', 2))
print(greet(42, 2))
print(greet.__annotations__)

Привет, Anna, Привет, Anna
Привет, 42, Привет, 42
{'name': <class 'str'>, 'times': <class 'int'>, 'return': <class 'str'>}


In [53]:
from collections.abc import Iterable
def average(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)

def index_by_name(users: Iterable[dict[str, str]]) -> dict[str, dict[str, str]]:
    return {u['name']: u for u in users}

Pount2D = tuple[float, float]

def distance(a: Pount2D, b: Pount2D) -> float:
    return ...



In [ ]:
def first(items: list[T]) -> T:
    return items[0]

class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []
    
    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()



In [ ]:
# протокол контейнера
class Playlist:
    def __init__(self, name, songs):
        self.name = name
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, index):
        return self._songs[index]


p = Playlist('road', ['Song A', 'Song B', 'Song C'])
print(len(p), p[0], p[-1], p[1:])
print('Song B' in p)
for song in p:
    print(song)
print(list(reversed(p)))
print(bool(Playlist('empty', [])))


# Python не проверяет, наследуется ли Playlist от list. Если объект ведет себя как последовательность, он и есть последовательность.
class Playlist:
    def __init__(self, name, songs):
        self.name = name
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __iter__(self):
        print('  вызван __iter__')
        yield from self._songs

    def __contains__(self, song):
        print('  вызван __contains__')
        return song.lower() in (s.lower() for s in self._songs)


p = Playlist('road', ['Song A', 'Song B'])
print('song b' in p)
print(list(p))

In [56]:
from typing import Protocol, Literal

class Sized(Protocol):
    def __len__(self) -> int: ...

def is_big(obj: Sized, limit: int = 2) -> bool:
    return len(obj) > limit

class Playlist:
    def __init__(self, songs: list[str]) -> None:
        self._songs = songs

    def __len__(self) -> int:
        return len(self._songs)

print(is_big([1, 2, 3]), is_big('ab'), is_big(Playlist(['a', 'b', 'w'])))

Mode = Literal['r', 'w', 'a']

def open_log(path: str, mode: Mode = 'r') -> str:
    return f'{path}:{mode}'

print(open_log('app.log', 's'))

True False True
app.log:s
